# NB_CursorUsage_Gold
**Source:** `AI_Medallion.Silver.CursorUsage`  
**Target:** `AI_Medallion.Gold.FactCursorActive`  
**Grain:** User + Date (one fact row per silver usage event)  
**Load type:** Incremental (Delta merge on `RecordSurrogateKey`)

In [5]:
%run ./common/bootstrap


Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set MEDALLION_CACHE_REFRESH=1 to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAKE_WRITE=pyspark to use JVM abfss (often hangs).


In [6]:
silver = read_table(TABLE_CURSOR_SILVER)

request = F.lit(1).cast("int")
use_case_multiplier = F.lit(CURSOR_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(CURSOR_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(CURSOR_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")

read_table: cache hit AI_Medallion.Silver.cursor_usage


In [ ]:
apply_local_merge_spark_conf(spark)

fact = silver.select(
    F.col("CursorUsageMonthNo").alias("Month_No"),
    request.alias("Request"),
    F.lit(CURSOR_ASSET_ID).alias("assetId"),
    F.lit(CURSOR_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("OktaUserId").alias("userId"),
    F.col("OktaUserFullName").alias("username"),
    F.col("OktaUserEmployeeNumber").alias("employeeid"),
    F.col("OktaUserEmail").alias("userEmail"),
    F.col("OktaUserFullEmail").alias("userFullEmail"),
    F.col("OktaUserRegion").alias("Division"),
    F.col("OktaUserCountry").alias("userCountry"),
    F.lit(CURSOR_USE_CASE_CODE).alias("useCaseCode"),
    F.lit(CURSOR_USE_CASE_DISPLAY_NAME).alias("useCaseDisplayName"),
    (request.cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier).alias(
        "UseCaseHoursSaved"
    ),
    (
        request.cast("decimal(18,8)")
        * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("CursorUsageDate").alias("LastModification"),
    F.col("HCBusinessUnit").alias("User_UN"),
    F.col("HCJobTitle").alias("JobRoles"),
    F.col("HCOktaUserCountry").alias("Employee_Country"),
    F.col("HCOffice").alias("Employee_Office"),
    F.col("HCTalentSegment").alias("Segment"),
    F.col("HCDivision").alias("Foundation"),
    F.col("HCBusinessUnitCode").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(fact_cursor_active_MERGE_KEY),
)

# Required not-null columns per spec
fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(fact_cursor_active_MERGE_KEY).isNotNull()
)

publish_merge_staging(fact, TABLE_fact_cursor_active)


publish_merge_staging: AI_Medallion.Gold.fact_cursor_active -> C:\Users\abaitali\.fabric\merge_staging\AI_Medallion__Gold__fact_cursor_active


WindowsPath('C:/Users/abaitali/.fabric/merge_staging/AI_Medallion__Gold__fact_cursor_active')

In [ ]:
merge_incremental(
    spark,
    fact,
    TABLE_fact_cursor_active,
    fact_cursor_active_MERGE_KEY,
)

show_sample(fact.limit(10))

merge_incremental: deltalake path abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Gold/Fact_Cursor_Active
merge_incremental: chunk 1 (22720 rows)...
merge_incremental: deltalake merge done


DataFrame[Month_No: string, Request: int, assetId: string, assetName: string, eventType: string, id_project: string, project: string, account: string, userId: string, username: string, employeeid: string, userEmail: string, userFullEmail: string, Division: string, userCountry: string, useCaseCode: string, useCaseDisplayName: string, UseCaseHoursSaved: decimal(38,6), UseCaseTotalHours: decimal(38,9), UserWithActivity: int, LastModification: string, User_UN: string, JobRoles: string, Employee_Country: string, Employee_Office: string, Segment: string, Foundation: string, Employee_Business_Unit_Code: string, asset_model: string, asset_useCase: string, asset_mySourceType: string, asset_sourceType: string, billingClientName: string, billingCountry: string, billingRegion: string, flowid: string, useCaseMultiplier: decimal(18,8), automation_poc: decimal(18,8), totalHoursSaved_poc: decimal(18,8), RecordSurrogateKey: string]